# HU-05 — Análisis de Valores Atípicos (Outlier Analysis)

> **PredictMain · CRISP-DM Fase: Preparación de Datos**

Este notebook es el documento ejecutable de revisión de dominio para el análisis de outliers de los sensores industriales del dataset AI4I 2020.  
Permite al validador del dominio reproducir las detecciones, revisar cada decisión y confirmar o cuestionar las justificaciones técnicas.

---

**Entradas:**  `data/processed/ai4i2020_clean.csv`  
**Salidas:**   `reports/outlier_report.md`, `data/processed/ai4i2020_outlier_analyzed.csv`

**Principio fundamental:** Los outliers **NO se eliminan automáticamente**. Cada decisión (KEEP / CAP / EXCLUDE) está justificada con criterios del dominio físico de la maquinaria.

## 1. Configuración del entorno

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Asegura que ml_service/src sea importable desde el notebook
ML_SERVICE = Path().resolve().parent  # ml_service/
if str(ML_SERVICE) not in sys.path:
    sys.path.insert(0, str(ML_SERVICE))

print(f"numpy version : {np.__version__}")
print(f"pandas version: {pd.__version__}")
print(f"ml_service root: {ML_SERVICE}")

numpy version : 2.5.4
pandas version: 3.1.1
ml_service root: C:\Users\user\Documents\SEPTIMO\edt_PredictMain\predictmain\ml_service


## 2. Carga del dataset limpio (HU-04 output)

In [2]:
from src.infrastructure.paths import paths

df = pd.read_csv(paths.clean_csv_path)
print(f"Shape: {df.shape}")
print(f"Nulos totales: {df.isnull().sum().sum()}")
df.head(3)

Shape: (10000, 14)
Nulos totales: 0


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0


## 3. Variables de sensores analizadas

Solo las 5 variables continuas operativas son candidatas para análisis de outliers.  
Los identificadores (`UDI`, `Product ID`) y las etiquetas binarias (`Machine failure`, modos de falla) **no aplican**.

In [3]:
SENSOR_COLS = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
]
df[SENSOR_COLS].describe().round(2)

,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min]
count,10000.0,10000.00,10000.00,10000.00,10000.00
mean,300.0,310.01,1538.78,39.99,107.95
std,2.0,1.48,179.28,9.97,63.65
min,295.3,305.70,1168.00,3.80,0.00
25%,298.3,308.80,1423.00,33.20,53.00
50%,300.1,310.10,1503.00,40.10,108.00
75%,301.5,311.10,1612.00,46.80,162.00
max,304.5,313.80,2886.00,76.60,253.00


## 4. Detección con IQR (1.5×) — `numpy.percentile`

Referencia: [numpy.percentile — NumPy v2 docs](https://numpy.org/doc/stable/reference/generated/numpy.percentile.html)

La regla IQR define el rango intercuartílico `IQR = Q3 - Q1` y las vallas:
- **Fence low**  = Q1 − 1.5 × IQR  
- **Fence high** = Q3 + 1.5 × IQR

Toda observación fuera de esas vallas es un **candidato a outlier** estadístico.

In [4]:
iqr_results = []

for col in SENSOR_COLS:
    arr = df[col].to_numpy(dtype=float)
    # np.percentile(a, q) — q en escala [0, 100]
    q1  = float(np.percentile(arr, 25, method="linear"))
    q3  = float(np.percentile(arr, 75, method="linear"))
    iqr = q3 - q1
    fence_lo = q1 - 1.5 * iqr
    fence_hi = q3 + 1.5 * iqr
    n_out = int(((df[col] < fence_lo) | (df[col] > fence_hi)).sum())

    iqr_results.append({
        "Variable": col, "Q1": q1, "Q3": q3, "IQR": iqr,
        "Fence Low": fence_lo, "Fence High": fence_hi, "IQR Outliers": n_out
    })

pd.DataFrame(iqr_results).set_index("Variable").round(2)

,Q1,Q3,IQR,Fence Low,Fence High,IQR Outliers
Variable,,,,,,
Air temperature [K],298.3,301.5,3.2,293.50,306.30,0
Process temperature [K],308.8,311.1,2.3,305.35,314.55,0
Rotational speed [rpm],1423.0,1612.0,189.0,1139.50,1895.50,418
Torque [Nm],33.2,46.8,13.6,12.80,67.20,69
Tool wear [min],53.0,162.0,109.0,-110.50,325.50,0


## 5. Detección con Percentiles P1/P99 — `numpy.percentile`

Complementa el IQR con un enfoque basado en la distribución empírica del propio dataset.

In [5]:
pct_results = []

for col in SENSOR_COLS:
    arr = df[col].to_numpy(dtype=float)
    p1  = float(np.percentile(arr, 1,  method="linear"))   # P1
    p99 = float(np.percentile(arr, 99, method="linear"))   # P99
    n_out = int(((df[col] < p1) | (df[col] > p99)).sum())

    pct_results.append({
        "Variable": col, "P1": p1, "P99": p99, "P1/P99 Outliers": n_out
    })

pd.DataFrame(pct_results).set_index("Variable").round(2)

,P1,P99,P1/P99 Outliers
Variable,,,
Air temperature [K],295.9,304.10,162
Process temperature [K],306.6,313.20,195
Rotational speed [rpm],1275.0,2188.01,198
Torque [Nm],16.7,62.80,198
Tool wear [min],0.0,222.00,99


## 6. Límites físicos UCI — verificación de rango operativo

Los límites operativos de ingeniería del dataset AI4I 2020 (fuente: ficha técnica UCI) son la referencia definitiva para distinguir entre:

| Tipo | Criterio |
|---|---|
| **Error de medición** | Fuera del rango físico UCI |
| **Extremo válido** | Dentro del rango UCI pero estadísticamente extremo |
| **Señal de falla** | Estadísticamente extremo y correlacionado con `Machine failure = 1` |

In [6]:
UCI_RANGES = {
    "Air temperature [K]":     (295.3, 304.5),
    "Process temperature [K]": (305.7, 313.8),
    "Rotational speed [rpm]":  (1168.0, 2886.0),
    "Torque [Nm]":             (3.8, 76.6),
    "Tool wear [min]":         (0.0, 253.0),
}

physical_check = []
for col, (lo, hi) in UCI_RANGES.items():
    n_out = int(((df[col] < lo) | (df[col] > hi)).sum())
    physical_check.append({"Variable": col, "UCI Min": lo, "UCI Max": hi,
                            "Fuera de rango físico": n_out,
                            "Veredicto": "✅ Ninguno" if n_out == 0 else f"⚠️ {n_out} registros"})

pd.DataFrame(physical_check).set_index("Variable")

,UCI Min,UCI Max,Fuera de rango físico,Veredicto
Variable,,,,
Air temperature [K],295.3,304.5,0,✅ Ninguno
Process temperature [K],305.7,313.8,0,✅ Ninguno
Rotational speed [rpm],1168.0,2886.0,0,✅ Ninguno
Torque [Nm],3.8,76.6,0,✅ Ninguno
Tool wear [min],0.0,253.0,0,✅ Ninguno


## 7. Correlación outliers IQR ↔ Machine failure

Antes de decidir qué hacer con los outliers estadísticos, verificamos si coinciden con eventos de falla real.  
Esto distingue **señales de falla** (KEEP obligatorio) de **errores de medición** (candidatos a CAP o EXCLUDE).

In [7]:
corr_results = []

for col in SENSOR_COLS:
    arr = df[col].to_numpy(dtype=float)
    q1  = float(np.percentile(arr, 25, method="linear"))
    q3  = float(np.percentile(arr, 75, method="linear"))
    iqr = q3 - q1
    fence_lo = q1 - 1.5 * iqr
    fence_hi = q3 + 1.5 * iqr

    mask_out = (df[col] < fence_lo) | (df[col] > fence_hi)
    n_outliers = mask_out.sum()
    n_failure_in_outliers = df.loc[mask_out, "Machine failure"].sum()
    pct = (n_failure_in_outliers / n_outliers * 100) if n_outliers > 0 else 0

    corr_results.append({
        "Variable": col,
        "IQR Outliers": int(n_outliers),
        "Con Machine failure=1": int(n_failure_in_outliers),
        "% Falla en outliers": f"{pct:.1f}%"
    })

pd.DataFrame(corr_results).set_index("Variable")

,IQR Outliers,Con Machine failure=1,% Falla en outliers
Variable,,,
Air temperature [K],0,0,0.0%
Process temperature [K],0,0,0.0%
Rotational speed [rpm],418,35,8.4%
Torque [Nm],69,62,89.9%
Tool wear [min],0,0,0.0%


## 8. Tabla de decisiones — Regla → Tipo → Decisión

Consolidación de todos los análisis anteriores en la tabla de decisión final.

In [8]:
from src.domain.outlier_rules import OUTLIER_RULES

decision_table = pd.DataFrame([{
    "Variable":        r.column,
    "Método":          r.detection_method,
    "Tipo de atípico": r.outlier_type,
    "Decisión":        r.decision,
    "Rango UCI":       f"[{r.uci_physical_min}, {r.uci_physical_max}]",
    "Justificación":   r.rationale[:80] + "..."
} for r in OUTLIER_RULES])

decision_table.set_index("Variable")

,Método,Tipo de atípico,Decisión,Rango UCI,Justificación
Variable,,,,,
Air temperature [K],IQR+PERCENTILE,valid_extreme,KEEP,"[295.3, 304.5]",Air temperature varies naturally with ambient ...
Process temperature [K],IQR+PERCENTILE,valid_extreme,KEEP,"[305.7, 313.8]",Process temperature reflects friction from the...
Rotational speed [rpm],IQR+PERCENTILE,failure_signal,KEEP,"[1168.0, 2886.0]",IQR (fence: 1139–1895 rpm) flags 418 records a...
Torque [Nm],IQR+PERCENTILE,failure_signal,KEEP,"[3.8, 76.6]",IQR (fence: 12.8–67.2 Nm) flags 69 records. Lo...
Tool wear [min],IQR+PERCENTILE,valid_extreme,KEEP,"[0.0, 253.0]","IQR fence is [-110.5, 325.5 min] — wider than ..."


## 9. Ejecución del pipeline completo

Reproduce los entregables oficiales de HU-05: `outlier_report.md` y `ai4i2020_outlier_analyzed.csv`.

In [9]:
from src.application.outlier_analysis import OutlierAnalysisService

service = OutlierAnalysisService()
result = service.run(
    csv_path=paths.clean_csv_path,
    output_csv_path=paths.outlier_csv_path,
    report_path=paths.outlier_report_path,
)

print(f"Registros entrada  : {result.rows_input:,}")
print(f"Outliers detectados: {result.total_detected:,}")
print(f"Valores modificados: {result.total_modified:,}")
print(f"Filas excluidas    : {result.total_excluded:,}")
print(f"Registros salida   : {result.rows_output:,}")

Registros entrada  : 10,000
Outliers detectados: 1,170
Valores modificados: 0
Filas excluidas    : 0
Registros salida   : 10,000


## 10. Resumen por columna

In [10]:
summary = pd.DataFrame([{
    "Variable":    s.column,
    "IQR outliers": s.iqr_outliers_count,
    "P1/P99 outliers": s.percentile_outliers_count,
    "Detectados (unión)": s.records_detected,
    "Modificados": s.records_modified,
    "Excluidos":   s.records_excluded,
    "Decisión":    s.decision,
} for s in result.column_stats])

summary.set_index("Variable")

,IQR outliers,P1/P99 outliers,Detectados (unión),Modificados,Excluidos,Decisión
Variable,,,,,,
Air temperature [K],0,162,162,0,0,KEEP
Process temperature [K],0,195,195,0,0,KEEP
Rotational speed [rpm],418,198,516,0,0,KEEP
Torque [Nm],69,198,198,0,0,KEEP
Tool wear [min],0,99,99,0,0,KEEP


## 11. Clasificación del dominio: error / extremo válido / señal de falla

El reporte técnico incorpora esta tabla con los conteos por tipo y los valores fuera de rango UCI.

In [11]:
type_labels = {
    "measurement_error": "Error de medición",
    "valid_extreme": "Valor extremo válido",
    "failure_signal": "Posible señal de falla",
}

classification = pd.DataFrame([
    {
        "Tipo": label,
        "Variables": sum(s.outlier_type == key for s in result.column_stats),
        "Registros detectados": sum(
            s.records_detected for s in result.column_stats
            if s.outlier_type == key
        ),
        "Fuera de rango UCI": sum(
            s.outside_physical_range for s in result.column_stats
            if s.outlier_type == key
        ),
    }
    for key, label in type_labels.items()
])

measurement_errors = sum(
    s.records_detected for s in result.column_stats
    if s.outlier_type == "measurement_error"
)
print(f"Errores de medición: {measurement_errors}")
classification.set_index("Tipo")

Errores de medición: 0


,Variables,Registros detectados,Fuera de rango UCI
Tipo,,,
Error de medición,0,0,0
Valor extremo válido,3,456,0
Posible señal de falla,2,714,0


## 12. Conclusión

| Hallazgo | Resultado |
|---|---|
| Variables analizadas | 5 sensores continuos |
| Total outliers detectados (IQR + P1/P99) | **1,170** |
| Fuera del rango físico UCI | **0** |
| Errores de medición | **0** |
| Decisión aplicada | **KEEP en todos los casos** |
| Valores modificados | **0** |
| Filas excluidas | **0** |

**Conclusión del dominio:** Todos los valores estadísticamente atípicos están dentro del rango físico documentado por UCI. Los outliers de RPM y Torque son señales de precursores de falla (PWF, OSF) con alto valor predictivo para el modelo. Eliminarlos introduciría sesgo y reduciría la capacidad de detección temprana.

---
*Notebook generado para revisión del validador de dominio — HU-05 PredictMain*